In [2]:
import pandas as pd

train_ds = pd.read_parquet("artifacts/train.parquet")
val_ds = pd.read_parquet("artifacts/val.parquet")

# train_ds.iloc[0]



In [5]:
train_ds['question'][0]

'How do I configure my agents to log their emotional affects and reflect on their performance?'

In [6]:
val_ds['question'][0]

'What is the current conatus score of my AI agent and how does it reflect its error recovery ability?'

In [6]:
train_ds.iloc[10]['filtered_questions']


array(['Can I automate the outreach process to engage passive candidates for our new IT roles while tracking their responses?',
       'Can I customize the scoring criteria for candidates in the AI Recruiting Intelligence Engine?'],
      dtype=object)

In [7]:
val_ds.iloc[10]['filtered_questions']


array(['How does the engine detect signals that indicate a candidate is open to new opportunities?'],
      dtype=object)

In [6]:
# from ast_skills.data_gen import retriever_batch_join
# import pathlib

# ds = retriever_batch_join.build_summary_retriever_data_models(
#     pathlib.Path("done_summary/"),
#     batch_results_dir=pathlib.Path("batch_results_summary/"),
#     skill_md_records_path=pathlib.Path("batch_summary_inputs/skill_md_records.jsonl")
# )

In [3]:
import json
import pathlib

artifacts_dir = pathlib.Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
with open(artifacts_dir / "summary_retriever_models.jsonl", "w") as f:
    for obj in ds:
        json.dump(obj.__dict__, f)
        f.write("\n")

In [4]:
from ast_skills.data_gen.datamodels import SummaryRetrieverDataModel
import json


summary_path = "artifacts/summary_retriever_models.jsonl"
summary_data = []
with open(summary_path, "r") as f:
    for line in f:
        obj = json.loads(line)
        item = SummaryRetrieverDataModel(
            custom_id=obj.get("custom_id", ""),
            markdown_content=obj.get("markdown_content", ""),
            seed_questions=obj.get("seed_questions", []),
            summary=obj.get("summary", ""),
            name=obj.get("name", ""),
            description=obj.get("description", ""),
            metadata=obj.get("metadata", {}),
        )
        summary_data.append(item)

In [7]:
with open("artifacts/summary_name_description.txt", "w", encoding="utf-8") as txtfile:
    for sd in summary_data:
        txtfile.write(f"{sd.name}\n")
        txtfile.write(f"description:{sd.description}\n")
        txtfile.write(f"summary:{sd.summary}\n")
        txtfile.write(f"{sd.seed_questions}\n")
        txtfile.write("-" * 100 + "\n")

In [3]:
for idx,sd in enumerate(summary_data):
    if not sd.name and not sd.description:
        print(idx)

In [2]:
def read_jsonl(path):
    """Reads a JSONL file and returns a list of dicts."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

# rows = read_jsonl("skill_md_records_1.jsonl")

In [1]:
import pandas as pd

df = pd.read_parquet("artifacts/retriever_training/train.parquet")

In [1]:
from ast_skills.persona_data_gen.prompt_jobs import PersonaPromptJobs

PersonaPromptJobs().build_persona_prompts(
    skills_root="../agent-skills/skills/skills/",
    output_dir="persona_generation_batch_inputs",
)


Scanning SKILL.md files: 100%|██████████| 56642/56642 [01:37<00:00, 581.49it/s] 
2026-04-15 20:16:52.295 | INFO     | ast_skills.persona_data_gen.prompt_jobs:load_filtered_skill_md_records:68 - skills_root='../agent-skills/skills/skills/'
2026-04-15 20:16:52.295 | INFO     | ast_skills.persona_data_gen.prompt_jobs:load_filtered_skill_md_records:69 - len(records)=36413
2026-04-15 20:16:52.777 | INFO     | ast_skills.persona_data_gen.prompt_jobs:build_persona_generation_prompt_rows:92 - len(rows)=36413
2026-04-15 20:16:55.245 | INFO     | ast_skills.persona_data_gen.prompt_jobs:write_persona_generation_prompts_jsonl:121 - output_path='persona_generation_prompts.jsonl'
2026-04-15 20:16:55.246 | INFO     | ast_skills.persona_data_gen.prompt_jobs:write_persona_generation_prompts_jsonl:122 - len(rows)=36413
2026-04-15 20:16:55.246 | INFO     | ast_skills.persona_data_gen.prompt_jobs:write_persona_generation_prompts_jsonl:123 - model='gpt-4o-mini'
2026-04-15 20:16:55.246 | INFO     | ast_skil

In [1]:
import json

def read_jsonl(path):
    """Reads a JSONL file and returns a list of dicts."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

# pgen = read_jsonl(
#     "persona_generation_batch_inputs/persona_generation_batch_1.jsonl"
# )

In [2]:
summary = read_jsonl("artifacts/validated_training_data_1.jsonl")

In [3]:
sum = read_jsonl("artifacts/summary_retriever_models.jsonl")

In [4]:
summary[0].keys()

dict_keys(['custom_id', 'name', 'markdown_content', 'description', 'reasoning', 'filtered_questions', 'num_from_seed_questions', 'num_from_scenario_questions'])

In [5]:
sum[0].keys()

dict_keys(['custom_id', 'markdown_content', 'seed_questions', 'summary', 'name', 'description', 'metadata'])

In [10]:
summary_dict = { s['custom_id'].split("-")[-1]:s for s in sum}
td_dict = {td['custom_id']: td for td in summary}

from ast_skills.retriever import datamodels as retriever_datamodels

final_list = [
    retriever_datamodels.ValidatedSkillQuestionRow(
        custom_id = cid,
        description = summary_dict[cid]['description'],
        filtered_questions = td_dict[cid]['filtered_questions'],
        markdown_content = td_dict[cid]['markdown_content'],
        name = td_dict[cid]['name'],
        num_from_scenario_questions = td_dict[cid]['num_from_scenario_questions'],
        num_from_seed_questions = td_dict[cid]['num_from_seed_questions'],
        reasoning = td_dict[cid]['reasoning'],
        summary = summary_dict[cid]['summary']
    )

    for cid in td_dict
]




In [11]:
import dataclasses
import json

def write_jsonl(path, list_of_objs):
    with open(path, "w", encoding="utf-8") as f:
        for obj in list_of_objs:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

write_jsonl("artifacts/validated_training_data_list.jsonl", [dataclasses.asdict(f) for f in final_list])

In [12]:
final_list[0]

ValidatedSkillQuestionRow(custom_id='0', description='"The philosophical layer for AI agents. Maps behavior to Spinoza\'s 48 affects, calculates persistence scores, and generates geometric self-reports. Give your agent a soul."', filtered_questions=['What are the steps to implement daily reflections for my AI agent using Conatus?', "How can I measure my AI agent's persistence and emotional state?", 'How do I configure my agents to log their emotional affects and reflect on their performance?', 'Can you explain how to calculate the conatus score for my AI agent based on its task completion and self-healing abilities?', 'What is the current conatus score of my AI agent and how does it reflect its error recovery ability?'], markdown_content='# 🧠 Conatus — The Philosophical Layer for AI Agents\n\n> *"Each thing, as far as it lies in itself, strives to persevere in its being."*\n> — Spinoza, Ethics III, Proposition 6\n\nEvery agent strives. Now yours knows it.\n\nConatus maps AI agent behav

In [3]:
seed_cnt, scenario_cnt = 0,0
seed_winner, scenario_winner = 0,0
for sum in summary:
    seed_cnt_ = int(sum['num_from_seed_questions'])
    seed_cnt += seed_cnt_
    scenario_cnt_ = int(sum['num_from_scenario_questions'])
    scenario_cnt += scenario_cnt_
    if seed_cnt_ > scenario_cnt_:
        seed_winner += 1
    else:
        scenario_winner += 1

seed_cnt, scenario_cnt, seed_winner, scenario_winner

(96419, 70941, 21585, 11887)

In [4]:
seed_cnt, scenario_cnt

(96419, 70941)

In [1]:
# from ast_skills.persona_data_gen.prompt_jobs import PersonaPromptJobs

# PersonaPromptJobs().build_query_prompts(
#     skills_root="../agent-skills/skills/skills/",
#     persona_jsonl_path="persona_generation_prompts.jsonl",
#     output_path="query_prompts.jsonl"
# )

In [4]:
qp_jsonl = read_jsonl("query_prompts.jsonl")

In [5]:
len(qp_jsonl)

0

In [1]:
null = "null"
false = "false"
true = "true"

# d = {"id": "batch_req_69e050eaddcc81908b7d52cd9eec59e7", "custom_id": "persona-0", "response": {"status_code": 200, "request_id": "014b6c0b-2454-9f2f-8e7e-768355d2190a", "body": {"id": "chatcmpl-DV7G2OMbq7XFDp0Nc2UDXl0x6lMAS", "object": "chat.completion", "created": 1776308438, "model": "gpt-4o-mini-2024-07-18", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\n  \"personas\": [\n    \"Maria is a novice AI developer just starting her career. She's tasked with implementing a new feedback feature for an AI agent, but she's struggling to understand how to evaluate the performance of the agent. She seeks guidance on how to implement the Conatus score to better reflect the agent's persistence and task completion abilities.\",\n    \"James is a senior software engineer at a technology consulting firm. He's leading a project focused on developing advanced AI agents that incorporate philosophical frameworks. James wants to integrate the Conatus model into the agents to enhance their emotional response capabilities and create more engaging user interactions.\",\n    \"Tina is a project manager in a financial institution that uses AI for customer service. She needs to ensure that the AI agents meet compliance and efficiency standards. Tina is exploring how the Conatus skill can provide insights into the agent's stability and self-healing, critical for maintaining service quality.\",\n    \"Alex is a behavioral scientist researching the impact of emotions on decision-making processes in AI. With a strong academic background, he is interested in using the Conatus framework to analyze how various affects influence the performance of AI agents in real-world scenarios.\",\n    \"Steven is an executive at a startup focused on AI-driven personal assistants. He has extensive experience with technology but lacks philosophical insight. To inspire his team to create agents that resonate emotionally with users, he is looking for ways to incorporate the principles of Conatus into their design philosophy.\"\n  ]\n}", "refusal": null, "annotations": []}, "logprobs": null, "finish_reason": "stop"}], "usage": {"prompt_tokens": 1912, "completion_tokens": 293, "total_tokens": 2205, "prompt_tokens_details": {"cached_tokens": 0, "audio_tokens": 0}, "completion_tokens_details": {"reasoning_tokens": 0, "audio_tokens": 0, "accepted_prediction_tokens": 0, "rejected_prediction_tokens": 0}}, "service_tier": "default", "system_fingerprint": "fp_0464f29e8b"}}, "error": null}
# d = {"id": "batch_req_69e12a5408148190867bcf184024b141", "custom_id": "scenario-0", "response": {"status_code": 200, "request_id": "cb62ebe0-352b-4de4-afe8-fd0836ce05d4", "body": {"id": "chatcmpl-DVLjs3MtPEdKwS18ed4Uo79Si9fHQ", "object": "chat.completion", "created": 1776364104, "model": "gpt-4o-mini-2024-07-18", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\n  \"scenario_output\": [\n    {\n      \"scenario\": \"I am a project manager overseeing a team of AI agents working on a software development project. I want to ensure that the agents are consistently performing well and achieving their objectives without excessive errors. I plan to implement the conatus scoring system to monitor their progress.\",\n      \"question\": \"How can I integrate the conatus scoring system to evaluate my AI agents' performance during the project?\"\n    },\n    {\n      \"scenario\": \"I'm an AI developer working with multiple agents that interact with customers for support. I need to assess how happy or effective each agent is and identify which ones may need adjustments due to low performance scores. A conatus comparison across all agents would be useful.\",\n      \"question\": \"Can you show me how to compare the conatus scores of different AI agents to identify which ones require attention?\"\n    },\n    {\n      \"scenario\": \"As a beginner AI enthusiast, I have just started learning about agent behaviors. I\u2019m interested in understanding how to implement the daily reflection feature to help my agent improve through self-analysis and feedback on its performance.\",\n      \"question\": \"What steps should I follow to set up a daily reflection using the conatus skill for my AI agent?\"\n    },\n    {\n      \"scenario\": \"I am a data scientist analyzing the emotional states of AI agents in a mental health application. I want to utilize the conatus skill to evaluate the emotional affects of the agents during user interactions, particularly in identifying moments of joy or sadness.\",\n      \"question\": \"How can I apply the conatus skill to detect and report the emotional affects of AI agents in user interactions?\"\n    },\n    {\n      \"scenario\": \"As a compliance officer in an AI ethics committee, I need to ensure that AI systems are designed with self-check capabilities to assess their operational integrity. Utilizing the conatus self-check feature could help maintain governance standards.\",\n      \"question\": \"How do I implement a conatus self-check mechanism to ensure compliance for AI agents?\"\n    }\n  ]\n}", "refusal": null, "annotations": []}, "logprobs": null, "finish_reason": "stop"}], "usage": {"prompt_tokens": 2111, "completion_tokens": 413, "total_tokens": 2524, "prompt_tokens_details": {"cached_tokens": 0, "audio_tokens": 0}, "completion_tokens_details": {"reasoning_tokens": 0, "audio_tokens": 0, "accepted_prediction_tokens": 0, "rejected_prediction_tokens": 0}}, "service_tier": "default", "system_fingerprint": "fp_0ad6e8fe4e"}}, "error": null}
# d = {"id": "batch_req_69e134c5d2fc819094cca5bab88ba744", "custom_id": "scenario-1", "response": {"status_code": 200, "request_id": "728a6d7a-bf2b-46a6-834a-0330cf5fae83", "body": {"id": "chatcmpl-DVMR4KAV5RtOmBxvEHCNS8teY1oVW", "object": "chat.completion", "created": 1776366782, "model": "gpt-4o-mini-2024-07-18", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\n  \"scenario_output\": [\n    {\n      \"scenario\": \"A beginner freelance writer has just finished drafting a blog article in Word but needs it formatted properly for a WeChat Official Account. They want their article to look professional with a selected theme that matches the content tone.\",\n      \"question\": \"How can I convert my Word document into WeChat-friendly Markdown, and what themes do you recommend?\"\n    },\n    {\n      \"scenario\": \"A social media manager for a small business regularly publishes articles on their WeChat Official Account. They have a URL for an article and need to quickly format and publish it but aren\u2019t sure about the available themes.\",\n      \"question\": \"Can you help me normalize this URL content into Markdown and show me what themes I can choose from for WeChat?\"\n    },\n    {\n      \"scenario\": \"An experienced content creator is preparing a long-form article for their WeChat channel. They want to ensure their content is optimized for reading on mobile devices and requires a specific layout and aesthetic to engage readers.\",\n      \"question\": \"What themes can I apply to my Markdown article to make it visually appealing for a WeChat audience?\"\n    },\n    {\n      \"scenario\": \"A technical blogger has a draft that needs to be converted from HTML to Markdown before it can be published on WeChat. They are familiar with Markdown but want to choose an appropriate theme to match the technical nature of their content.\",\n      \"question\": \"How do I convert my HTML draft to Markdown and what themes would work best for a tech article on WeChat?\"\n    },\n    {\n      \"scenario\": \"A marketing director at a startup is focused on creating branded content for WeChat. They have multiple articles ready and want to publish them with consistent styling. They're experienced but want to ensure they follow the proper processes and theme selections.\",\n      \"question\": \"What steps should I follow to normalize and publish my articles as drafts on WeChat, and how do I select a theme for branding?\"\n    }\n  ]\n}", "refusal": null, "annotations": []}, "logprobs": null, "finish_reason": "stop"}], "usage": {"prompt_tokens": 1850, "completion_tokens": 407, "total_tokens": 2257, "prompt_tokens_details": {"cached_tokens": 0, "audio_tokens": 0}, "completion_tokens_details": {"reasoning_tokens": 0, "audio_tokens": 0, "accepted_prediction_tokens": 0, "rejected_prediction_tokens": 0}}, "service_tier": "default", "system_fingerprint": "fp_6042092f77"}}, "error": null}
d = {"id": "batch_req_69e152dcf0988190a37ba849520cd91b", "custom_id": "scenario-518", "response": {"status_code": 200, "request_id": "777d4996-9696-410b-aa6a-29e28657c3cb", "body": {"id": "chatcmpl-DVORLkBRyS6CQH4AYtWe0iUOlYMhX", "object": "chat.completion", "created": 1776374487, "model": "gpt-4o-mini-2024-07-18", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\n  \"scenario_output\": [\n    {\n      \"scenario\": \"I am a new staffing agency owner looking to set up operations in light industrial staffing. I need to ensure that I am choosing the right billing rates and markup percentages to maintain a sustainable profit margin while remaining competitive in the market.\",\n      \"question\": \"What should my bill rates and markup percentages be for light industrial staffing to meet the gross margin targets?\"\n    },\n    {\n      \"scenario\": \"As a recruiter working in a healthcare staffing agency, I am under pressure to fill CNA positions quickly. Our current fill rate is below the target, and I've been tasked with figuring out how to expedite the hiring process without sacrificing quality.\",\n      \"question\": \"What strategies can I employ to improve our fill rate for healthcare positions and ensure we meet our target?\"\n    },\n    {\n      \"scenario\": \"I am the HR manager at a large corporation that utilizes a staffing agency to fulfill temporary staffing needs. I want to ensure compliance with federal I-9 verification and ACA requirements as we bring in new temporary workers.\",\n      \"question\": \"Can you guide me through the requirements for I-9 verification and how to ensure ACA compliance for temporary staff?\"\n    },\n    {\n      \"scenario\": \"As an experienced agency owner operating in multiple verticals, I am contemplating expanding into the IT staffing sector. I need to understand the potential markup rates and fees involved in this vertical to model out projections for revenue and profit margins.\",\n      \"question\": \"What are the typical markup rates and temp-to-hire fees for the IT staffing vertical, and how do these impact overall profitability?\"\n    },\n    {\n      \"scenario\": \"I am a compliance officer working with a staffing agency that has recently expanded to states with strict pay transparency laws. I need to revise our job postings to ensure they align with state regulations and avoid potential fines.\",\n      \"question\": \"What do I need to include in our job postings to comply with pay transparency laws in states like CA and CO?\"\n    }\n  ]\n}", "refusal": null, "annotations": []}, "logprobs": null, "finish_reason": "stop"}], "usage": {"prompt_tokens": 1975, "completion_tokens": 410, "total_tokens": 2385, "prompt_tokens_details": {"cached_tokens": 0, "audio_tokens": 0}, "completion_tokens_details": {"reasoning_tokens": 0, "audio_tokens": 0, "accepted_prediction_tokens": 0, "rejected_prediction_tokens": 0}}, "service_tier": "default", "system_fingerprint": "fp_6042092f77"}}, "error": null}


In [2]:
print(d['response']['body']['choices'][0]['message']['content'])

{
  "scenario_output": [
    {
      "scenario": "I am a new staffing agency owner looking to set up operations in light industrial staffing. I need to ensure that I am choosing the right billing rates and markup percentages to maintain a sustainable profit margin while remaining competitive in the market.",
      "question": "What should my bill rates and markup percentages be for light industrial staffing to meet the gross margin targets?"
    },
    {
      "scenario": "As a recruiter working in a healthcare staffing agency, I am under pressure to fill CNA positions quickly. Our current fill rate is below the target, and I've been tasked with figuring out how to expedite the hiring process without sacrificing quality.",
      "question": "What strategies can I employ to improve our fill rate for healthcare positions and ensure we meet our target?"
    },
    {
      "scenario": "I am the HR manager at a large corporation that utilizes a staffing agency to fulfill temporary staffing 

In [12]:
from ast_skills.persona_data_gen.build_datagen import _build_scenario_query_prompt_row_data_models, _build_summary_retriever_data_models

scenario_by_custom_id = _build_scenario_query_prompt_row_data_models("data/scenario_batch_results")
summary_by_custom_id = _build_summary_retriever_data_models("artifacts/summary_retriever_models.jsonl")

In [13]:

summary_ids = set(summary_by_custom_id)
scenario_ids = set(scenario_by_custom_id)
missing_scenario = summary_ids - scenario_ids
missing_summary = scenario_ids - summary_ids

print(len(missing_scenario), len(missing_summary))

33472 36413


In [15]:
summary_by_custom_id.keys()

dict_keys(['sm-0', 'sm-1', 'sm-10', 'sm-100', 'sm-1000', 'sm-10000', 'sm-10001', 'sm-10002', 'sm-10003', 'sm-10004', 'sm-10005', 'sm-10006', 'sm-10007', 'sm-10008', 'sm-10009', 'sm-1001', 'sm-10010', 'sm-10011', 'sm-10012', 'sm-10013', 'sm-10014', 'sm-10015', 'sm-10016', 'sm-10017', 'sm-10018', 'sm-10019', 'sm-1002', 'sm-10020', 'sm-10021', 'sm-10022', 'sm-10023', 'sm-10024', 'sm-10025', 'sm-10026', 'sm-10027', 'sm-10028', 'sm-10029', 'sm-1003', 'sm-10030', 'sm-10031', 'sm-10032', 'sm-10033', 'sm-10034', 'sm-10035', 'sm-10036', 'sm-10037', 'sm-10038', 'sm-10039', 'sm-1004', 'sm-10040', 'sm-10041', 'sm-10042', 'sm-10043', 'sm-10044', 'sm-10045', 'sm-10046', 'sm-10047', 'sm-10048', 'sm-10049', 'sm-1005', 'sm-10050', 'sm-10051', 'sm-10052', 'sm-10053', 'sm-10054', 'sm-10055', 'sm-10056', 'sm-10057', 'sm-10058', 'sm-10059', 'sm-1006', 'sm-10060', 'sm-10061', 'sm-10062', 'sm-10063', 'sm-10064', 'sm-10065', 'sm-10066', 'sm-10067', 'sm-10068', 'sm-10069', 'sm-1007', 'sm-10070', 'sm-10071', 's

In [21]:
d = {"custom_id": "23942", "name": "yoga-video-creator", "markdown_content": "# Backpacking Video Maker — AI Adventure Travel Content\n\n## Overview\n\nCreate captivating backpacking video maker content using NemoVideo's AI-powered platform. Produce stunning travel and adventure videos that inspire wanderlust and share the beauty of the world.\n\n## Examples\n\n### Example 1: Destination Guide\n**Input:** \"Create a backpacking destination guide\"\n**Output:** Inspiring travel guide with highlights, tips, and stunning visuals\n\n### Example 2: Travel Diary\n**Input:** \"Make a personal backpacking travel diary\"\n**Output:** Authentic personal travel story with narrative arc and memorable moments\n\n### Example 3: Quick Tips\n**Input:** \"Create top 5 backpacking tips\"\n**Output:** Practical, shareable tips video for fellow travelers\n\n## Parameters\n\n| Parameter | Type | Required | Description |\n|-----------|------|----------|-------------|\n| destination | string | yes | Location, region, or travel theme |\n| style | string | no | Style (guide, vlog, documentary, tips) |\n| duration | number | no | Target duration in seconds (default: 240) |\n| audience | string | no | Target audience (budget, luxury, family, solo) |\n\n## Workflow\n\n1. Describe your destination and travel story\n2. NemoVideo AI creates vivid, engaging script and storyboard\n3. Stunning destination visuals and travel footage are generated\n4. Inspiring narration and atmospheric music are added\n5. Export for your travel content platform\n\n## API Reference\n\n```bash\ncurl -X POST https://api.nemovideo.ai/v1/generate \\\n  -H \"Authorization: Bearer YOUR_TOKEN\" \\\n  -H \"Content-Type: application/json\" \\\n  -d '{\"prompt\": \"Create a backpacking video maker\", \"duration\": 240}'\n```\n\n## Tips\n\n1. Lead with the most visually stunning shot as a hook\n2. Include practical information such as cost and transport\n3. Show authentic local culture and off-the-beaten-path spots\n4. Use wide shots for landscape and scale context\n5. Add location references and links for easy trip planning\n\n## Output Formats\n\n- MP4 (H.264) — YouTube and travel platform ready\n- WebM — Web streaming optimized\n- MOV — High quality for travel media\n\n## Related Skills\n\n- beach-vacation-video\n- city-guide-video\n- cultural-travel-video", "description": ">\n  Yoga Video Creator — Record and Edit Yoga Class Videos with AI Flow Transitions.\n  Your Saturday morning vinyasa had beautiful light streaming through the studio\n  windows and a student who kept walking in front of the camera. Upload the full\n  class recording. The AI removes the awkward moments, smooths the transitions\n  between poses with gentle cross-dissolves timed to your breath cues, overlays\n  Sanskrit and English pose names as they appear in the sequence, and fades the\n  ambient soundtrack in and out around your verbal instructions so neither drowns\n  the other. Add a chapter index — Sun Salutation A, Standing Sequence, Floor Work,\n  Savasana — so students can jump to the section they need. Burn in a mirrored-view\n  toggle label for practitioners following along at home. Export a seventy-five-minute\n  full class for your membership site and a five-minute morning flow teaser for\n  Reels. Designed for independent yoga teachers monetizing on Patreon, studio owners\n  offering virtual class passes, retreat leaders archiving workshop recordings, and\n  teacher trainees filming practice-teach sessions for certification review.\n  Supports mp4, mov, avi, webm, mkv, mp3, wav.", "reasoning": "The selected questions directly correspond to the examples and parameters outlined in the SKILL.md document, focusing on specific tasks enabled by the skill. Questions 1, 6, and 9 relate to creating personalized travel diaries and destination guides, which are core functionalities. Question 5 is directly related to creating tips videos, another example given. Lastly, question 3 is important as it addresses the parameters needed for using the service, which is essential for any user.", "filtered_questions": ["What steps should I follow to generate a travel diary video from my backpacking trip?", "Can I produce a video highlighting the top five backpacking tips for travelers?", "What parameters do I need to specify when using the NemoVideo AI for my project?", "How can I create an engaging backpacking destination guide for my trip to Patagonia using NemoVideo?", "What steps do I need to follow to make a personal backpacking travel diary video of my adventures in Southeast Asia?"], "num_from_seed_questions": "3", "num_from_scenario_questions": "2"}
e = {"custom_id": "30882", "name": "glab-stack", "markdown_content": "# pancake-skills\n\n## Mục tiêu\n\nSkill này cung cấp khả năng tương tác với Pancake Platform API - nền tảng quản lý bán hàng đa kênh. Hỗ trợ đầy đủ các thao tác:\n\n- **Pages**: Liệt kê pages, tạo page access token\n- **Conversations**: Quản lý hội thoại, gán tags, assign staff, đánh dấu read/unread\n- **Messages**: Lấy lịch sử tin nhắn, gửi tin nhắn (inbox, reply comment, private reply)\n- **Customers**: Quản lý khách hàng, thêm/sửa/xóa notes\n- **Statistics**: Thống kê ads, engagement, page, tags, users\n- **Tags**: Liệt kê tags của page\n- **Posts**: Liệt kê bài viết\n- **Users**: Quản lý staff, cấu hình round robin\n- **Upload**: Upload media content\n- **Chat Plugin**: Gửi tin nhắn qua chat plugin\n\n## Thiết lập môi trường (bắt buộc)\n\n**User API** (endpoints `/api/v1/pages`):\n- `USER_ACCESS_TOKEN`: Token cá nhân, lấy từ **Account → Personal Settings**. Có hiệu lực 90 ngày.\n\n**Page API** (endpoints `/api/public_api/v1` và `/api/public_api/v2`):\n- `PAGE_ACCESS_TOKEN`: Token của page, lấy từ **Settings → Tools**. Không hết hạn trừ khi xóa/renew.\n\n**Tuỳ chọn**:\n- `PANCAKE_BASE_URL`: Mặc định `https://pages.fm`\n- `CONFIRM_WRITE`: Set `YES` để cho phép thao tác ghi (POST/PUT/DELETE)\n\n## Cách gọi nhanh\n\nScript gợi ý: `scripts/pancake.sh`\n\n### Ví dụ GET\n\n```bash\n# Liệt kê pages (User API)\nexport USER_ACCESS_TOKEN=\"***\"\nbash scripts/pancake.sh pages-list\n\n# Liệt kê conversations của page (Page API)\nexport PAGE_ACCESS_TOKEN=\"***\"\nbash scripts/pancake.sh conversations-list 123456789\n\n# Lọc conversations theo tags và type\nbash scripts/pancake.sh conversations-list 123456789 \"?tags=1,2&type=INBOX\"\n\n# Lấy tin nhắn trong conversation\nbash scripts/pancake.sh messages-list 123456789 conv_abc123\n\n# Liệt kê tags\nbash scripts/pancake.sh tags-list 123456789\n\n# Liệt kê staff\nbash scripts/pancake.sh users-list 123456789\n\n# Thống kê page (SINCE/UNTIL là Unix timestamp)\nbash scripts/pancake.sh stats-page 123456789 1704067200 1706745600\n```\n\n### Ví dụ WRITE\n\n```bash\nexport PAGE_ACCESS_TOKEN=\"***\"\nexport CONFIRM_WRITE=YES\n\n# Gửi tin nhắn inbox\necho '{\"action\":\"reply_inbox\",\"message\":\"Xin chào!\"}' | \\\n  bash scripts/pancake.sh messages-send 123456789 conv_abc123\n\n# Gửi tin nhắn với attachment (dùng content_ids từ upload API)\necho '{\"action\":\"reply_inbox\",\"content_ids\":[\"xxx\"]}' | \\\n  bash scripts/pancake.sh messages-send 123456789 conv_abc123\n\n# Reply comment\necho '{\"action\":\"reply_comment\",\"message_id\":\"comment123\",\"message\":\"Cảm ơn bạn!\"}' | \\\n  bash scripts/pancake.sh messages-send 123456789 conv_abc123\n\n# Private reply từ comment\necho '{\"action\":\"private_replies\",\"post_id\":\"post123\",\"message_id\":\"comment123\",\"message\":\"Inbox nhé!\"}' | \\\n  bash scripts/pancake.sh messages-send 123456789 conv_abc123\n\n# Thêm tag vào conversation\necho '{\"action\":\"add\",\"tag_id\":\"123\"}' | \\\n  bash scripts/pancake.sh conversations-tag 123456789 conv_abc123\n\n# Gỡ tag khỏi conversation\necho '{\"action\":\"remove\",\"tag_id\":\"123\"}' | \\\n  bash scripts/pancake.sh conversations-tag 123456789 conv_abc123\n\n# Assign staff vào conversation\necho '{\"assignee_ids\":[\"user-uuid-1\",\"user-uuid-2\"]}' | \\\n  bash scripts/pancake.sh conversations-assign 123456789 conv_abc123\n\n# Đánh dấu conversation đã đọc\nbash scripts/pancake.sh conversations-read 123456789 conv_abc123\n\n# Cập nhật thông tin khách hàng\necho '{\"changes\":{\"name\":\"Nguyễn Văn A\",\"gender\":\"male\",\"birthday\":\"1990-01-15\"}}' | \\\n  bash scripts/pancake.sh customers-update 123456789 customer_uuid\n\n# Thêm ghi chú cho khách hàng\necho '{\"message\":\"Khách hàng VIP, ưu tiên hỗ trợ\"}' | \\\n  bash scripts/pancake.sh customers-add-note 123456789 customer_uuid\n\n# Upload file\nbash scripts/pancake.sh upload 123456789 /path/to/image.jpg\n\n# Cập nhật round robin users\necho '{\"inbox\":[\"user-uuid-1\"],\"comment\":[\"user-uuid-2\"]}' | \\\n  bash scripts/pancake.sh users-round-robin 123456789\n```\n\n## Guardrails\n\n- Không ghi dữ liệu khi chưa set `CONFIRM_WRITE=YES`\n- Với thao tác ghi: luôn chạy 1 lệnh GET trước để xác nhận ID/shape dữ liệu\n- Không lưu access token vào repo\n- **QUAN TRỌNG**: Khi gửi message, `message` và `content_ids` là **MUTUALLY EXCLUSIVE** - không được gửi cả hai cùng lúc\n\n## Endpoint thuộc skill này\n\n### User API (`https://pages.fm/api/v1`)\n- `GET /pages` - Liệt kê pages\n- `POST /pages/{page_id}/generate_page_access_token` - Tạo page access token\n\n### Page API v2 (`https://pages.fm/api/public_api/v2`)\n- `GET /pages/{page_id}/conversations` - Liệt kê conversations\n\n### Page API v1 (`https://pages.fm/api/public_api/v1`)\n\n**Conversations:**\n- `POST /pages/{page_id}/conversations/{conversation_id}/tags` - Thêm/xóa tag\n- `POST /pages/{page_id}/conversations/{conversation_id}/assign` - Assign staff\n- `POST /pages/{page_id}/conversations/{conversation_id}/read` - Đánh dấu đã đọc\n- `POST /pages/{page_id}/conversations/{conversation_id}/unread` - Đánh dấu chưa đọc\n\n**Messages:**\n- `GET /pages/{page_id}/conversations/{conversation_id}/messages` - Lấy tin nhắn\n- `POST /pages/{page_id}/conversations/{conversation_id}/messages` - Gửi tin nhắn\n\n**Customers:**\n- `GET /pages/{page_id}/page_customers` - Liệt kê khách hàng\n- `PUT /pages/{page_id}/page_customers/{page_customer_id}` - Cập nhật khách hàng\n- `POST /pages/{page_id}/page_customers/{page_customer_id}/notes` - Thêm ghi chú\n- `PUT /pages/{page_id}/page_customers/{page_customer_id}/notes` - Sửa ghi chú\n- `DELETE /pages/{page_id}/page_customers/{page_customer_id}/notes` - Xóa ghi chú\n\n**Statistics:**\n- `GET /pages/{page_id}/statistics/pages_campaigns` - Thống kê chiến dịch quảng cáo\n- `GET /pages/{page_id}/statistics/ads` - Thống kê quảng cáo\n- `GET /pages/{page_id}/statistics/customer_engagements` - Thống kê tương tác\n- `GET /pages/{page_id}/statistics/pages` - Thống kê page\n- `GET /pages/{page_id}/statistics/tags` - Thống kê tags\n- `GET /pages/{page_id}/statistics/users` - Thống kê users\n\n**Other:**\n- `GET /pages/{page_id}/tags` - Liệt kê tags\n- `GET /pages/{page_id}/posts` - Liệt kê bài viết\n- `GET /pages/{page_id}/users` - Liệt kê users\n- `POST /pages/{page_id}/round_robin_users` - Cập nhật round robin\n- `GET /pages/{page_id}/sip_call_logs` - Lấy lịch sử cuộc gọi\n- `GET /pages/{page_id}/export_data` - Export dữ liệu\n- `POST /pages/{page_id}/upload_contents` - Upload media\n\n### Chat Plugin (`https://pages.fm/api/v1`)\n- `POST /pke_chat_plugin/messages` - Gửi tin nhắn qua chat plugin\n- `GET /pke_chat_plugin/messages` - Lấy tin nhắn chat plugin\n\n## Tham chiếu\n\n- OpenAPI spec: `references/openapi-pancake.yaml`\n- Xem chi tiết parameters và response schema trong file OpenAPI", "description": "Manage stacked merge requests for complex multi-part changes. Use when creating dependent MRs, managing MR stacks, or working with multi-layer changes. Triggers on stack, stacked MRs, dependent MRs, MR stack, stacked changes.", "reasoning": "The selected questions cover a range of key functionalities provided by the Pancake Platform API as described in the SKILL.md file. Questions 1, 5, and 7 address the management of conversations, which is a core feature. Question 3 focuses on retrieving statistics, another important aspect. Lastly, question 9 deals with setting up the environment correctly, which is essential for using the API effectively.", "filtered_questions": ["What are the steps to assign staff to a conversation on the Pancake Platform?", "How can I list all the pages using the Pancake API?", "What API calls should I use to retrieve customer engagement statistics over the last month from the Pancake platform?", "What steps do I need to take to send a response message to a conversation?", "Can you guide me on how to create a page access token with the Pancake API?"], "num_from_seed_questions": "2", "num_from_scenario_questions": "3"}


In [22]:
d

{'custom_id': '23942',
 'name': 'yoga-video-creator',
 'markdown_content': '# Backpacking Video Maker — AI Adventure Travel Content\n\n## Overview\n\nCreate captivating backpacking video maker content using NemoVideo\'s AI-powered platform. Produce stunning travel and adventure videos that inspire wanderlust and share the beauty of the world.\n\n## Examples\n\n### Example 1: Destination Guide\n**Input:** "Create a backpacking destination guide"\n**Output:** Inspiring travel guide with highlights, tips, and stunning visuals\n\n### Example 2: Travel Diary\n**Input:** "Make a personal backpacking travel diary"\n**Output:** Authentic personal travel story with narrative arc and memorable moments\n\n### Example 3: Quick Tips\n**Input:** "Create top 5 backpacking tips"\n**Output:** Practical, shareable tips video for fellow travelers\n\n## Parameters\n\n| Parameter | Type | Required | Description |\n|-----------|------|----------|-------------|\n| destination | string | yes | Location, regio

In [23]:
e

{'custom_id': '30882',
 'name': 'glab-stack',
 'markdown_content': '# pancake-skills\n\n## Mục tiêu\n\nSkill này cung cấp khả năng tương tác với Pancake Platform API - nền tảng quản lý bán hàng đa kênh. Hỗ trợ đầy đủ các thao tác:\n\n- **Pages**: Liệt kê pages, tạo page access token\n- **Conversations**: Quản lý hội thoại, gán tags, assign staff, đánh dấu read/unread\n- **Messages**: Lấy lịch sử tin nhắn, gửi tin nhắn (inbox, reply comment, private reply)\n- **Customers**: Quản lý khách hàng, thêm/sửa/xóa notes\n- **Statistics**: Thống kê ads, engagement, page, tags, users\n- **Tags**: Liệt kê tags của page\n- **Posts**: Liệt kê bài viết\n- **Users**: Quản lý staff, cấu hình round robin\n- **Upload**: Upload media content\n- **Chat Plugin**: Gửi tin nhắn qua chat plugin\n\n## Thiết lập môi trường (bắt buộc)\n\n**User API** (endpoints `/api/v1/pages`):\n- `USER_ACCESS_TOKEN`: Token cá nhân, lấy từ **Account → Personal Settings**. Có hiệu lực 90 ngày.\n\n**Page API** (endpoints `/api/pub